# EYES-DEFY-ANEMIA -- Phase 4 Classification -- new_way/ roster

8 architectures x 2 tissue types = 16 combos, added to the same shared Optuna engine
(`datapreparepipeline/trainer_engine.py`) and the same protocol as every prior combo in this
project (frozen ImageNet backbone, `Dropout->Linear` head, `dropout_rate`/`learning_rate`/
`weight_decay` all Optuna-tuned, 250-epoch ceiling, patience=7, 12-trial search) -- only the
architecture roster is new.

**5 CNN:** EfficientNet-B3 (10.70M), EfficientNet-B4 (17.55M), RegNetY-16GF (80.57M),
ConvNeXt-Base (87.57M), ConvNeXt-Large (196.23M).

**3 Hybrid:** MaxViT-Tiny (30.41M, torchvision), MaxViT-Small (68.16M, **timm**),
CoAtNet-3 (163.64M, **timm**).

**Dependency note:** MaxViT-Small and CoAtNet-3 do not exist in torchvision at all -- verified
directly (torchvision's only MaxViT variant is `maxvit_t`; CoAtNet was never ported to
torchvision in any size). `timm==1.0.28` was adopted as a real project dependency for this
reason (`classification/.project_memory/03_tech_stack_and_rules.md`), the first departure from
this project's original torchvision-only rule. **CoAtNet-3's pretrained weights are ImageNet-12k
only, with no ImageNet-1k fine-tuning stage** -- the one architecture in this roster with a
different pretraining regime than everything else in the project; worth noting explicitly if
reported alongside the other combos, not treated as an interchangeable backbone.

**Ordering:** cheapest-first by verified total parameter count, not by CNN/Hybrid family --
unlike the earlier CNN/ViT split, costs interleave here (ConvNeXt-Large at 196M is heavier than
any hybrid), so there's no clean family-based split into two sessions.

`sync_outputs()` runs after every combo, so an interrupted session still yields a downloadable
zip of everything completed so far.

## Setup

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [2]:
# rm -rf first so a re-run within the same kernel session stays idempotent
# instead of nesting a second clone inside the first (this bit an earlier
# version of this notebook during interactive debugging).
!rm -rf eyes-defy-anemia
!git clone https://github.com/manivafapour/eyes-defy-anemia.git
%cd eyes-defy-anemia

Cloning into 'eyes-defy-anemia'...
remote: Enumerating objects: 1073, done.
remote: Counting objects: 100% (754/754), done.
remote: Compressing objects: 100% (489/489), done.
remote: Total 1073 (delta 359), reused 628 (delta 259), pack-reused 319 (from 1)
Receiving objects: 100% (1073/1073), 76.39 MiB | 39.47 MiB/s, done.
Resolving deltas: 100% (522/522), done.
/kaggle/working/eyes-defy-anemia


In [3]:
# Diagnostic only -- kept for visibility in the saved run log. The path used
# below is already confirmed correct, so this cell doesn't gate anything, but
# it's cheap and makes a future path change easy to spot in the output.
import os

for name in os.listdir("/kaggle/input"):
    path = f"/kaggle/input/{name}"
    print(name, "->", os.listdir(path))

datasets -> ['manivafapour33']


In [4]:
# Only packages actually missing from Kaggle's base image. Deliberately NOT
# `pip install -r requirements.txt` -- that file is pinned to the local
# Windows/CUDA 13.0 build and would try to reinstall Kaggle's own correctly
# configured GPU PyTorch with an incompatible build.
#
# `timm` added here -- new for this notebook. MaxViT-Small and CoAtNet-3 do
# not exist in torchvision, so this roster is the first to need it on Kaggle too.
!pip install -q optuna albumentations timm

## Data

In [5]:
import shutil
from pathlib import Path

# TODO: verify against cell 4's /kaggle/input listing before running -- this is a best-guess
# default following the established manivafapour33/<slug> pattern, NOT yet confirmed for the
# new clean-data dataset (see classification/.project_memory/kaggle/01_kaggle_notes.md).
SRC_DIR = Path("/kaggle/input/datasets/manivafapour33/processed-dataset-clean")
DST_DIR = Path("classification/data/processed")

# Clear the destination first so this cell is fully idempotent -- a re-run (or
# the two messy copy attempts from the earlier notebook version) never leaves
# stale or duplicated content behind. DST_DIR always ends up as an exact,
# deterministic copy of SRC_DIR, nothing more.
shutil.rmtree(DST_DIR, ignore_errors=True)
DST_DIR.mkdir(parents=True, exist_ok=True)

for item in SRC_DIR.iterdir():
    dest = DST_DIR / item.name
    if item.is_dir():
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)

print("classification/data/processed now contains:")
for sub in sorted(DST_DIR.iterdir()):
    if sub.is_dir():
        n_files = sum(1 for f in sub.rglob("*") if f.is_file())
        print(f"  {sub.name}/  ({n_files} files)")
    else:
        print(f"  {sub.name}")

classification/data/processed now contains:
  extraction_log.csv
  images/  (428 files)
  metadata.csv
  splits.csv


## Registry sanity check

Confirms all 8 new architectures actually registered in the shared engine, and that each
builds, forward-passes, and produces the expected trainable-parameter count -- before any real
training starts. Mirrors the local structural verification already run for this roster.

In [6]:
import sys
sys.path.insert(0, "classification/datapreparepipeline")
import torch
from trainer_engine import ARCHITECTURE_REGISTRY, DEVICE

NEW_ARCHS = ['efficientnet_b3', 'efficientnet_b4', 'regnet_y_16gf', 'convnext_base',
             'convnext_large', 'maxvit_t', 'maxvit_small', 'coatnet_3']

for arch in NEW_ARCHS:
    cfg = ARCHITECTURE_REGISTRY[arch]
    model = cfg['build_fn'](0.2).to(DEVICE)
    x = torch.randn(2, 3, cfg['input_size'], cfg['input_size']).to(DEVICE)
    with torch.no_grad():
        out = model(x)
    assert out.shape == (2, 1), f'{arch}: bad output shape {out.shape}'
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'{arch:<20} OK  out={tuple(out.shape)}  trainable_params={n_trainable}')
    del model
print('\nAll 8 new_way architectures registered and working.')

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 156MB/s]


efficientnet_b3      OK  out=(2, 1)  trainable_params=1537
Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 217MB/s]


efficientnet_b4      OK  out=(2, 1)  trainable_params=1793
Downloading: "https://download.pytorch.org/models/regnet_y_16gf-9e6ed7dd.pth" to /root/.cache/torch/hub/checkpoints/regnet_y_16gf-9e6ed7dd.pth


...

maxvit_t             OK  out=(2, 1)  trainable_params=513


model.safetensors:   0%|          | 0.00/276M [00:00<?, ?B/s]

maxvit_small         OK  out=(2, 1)  trainable_params=769


model.safetensors:   0%|          | 0.00/727M [00:00<?, ?B/s]

coatnet_3            OK  out=(2, 1)  trainable_params=1537

All 8 new_way architectures registered and working.


## Output syncing

In [7]:
import shutil
from pathlib import Path


def sync_outputs():
    """Consolidate classification/outputs/{checkpoints,logs,plots}/ into a
    single top-level /kaggle/working/outputs/ folder and re-zip it to
    /kaggle/working/new_way_results.zip. Called after EVERY training cell --
    16 combos is a long unattended run, so whatever completed so far must
    always be downloadable, same pattern as every prior notebook here."""
    results_dir = Path("/kaggle/working/outputs")
    results_dir.mkdir(parents=True, exist_ok=True)
    for sub in ["checkpoints", "logs", "plots"]:
        src = Path("classification/outputs") / sub
        if src.exists():
            shutil.copytree(src, results_dir / sub, dirs_exist_ok=True)
    archive_path = shutil.make_archive("/kaggle/working/new_way_results", "zip", root_dir=str(results_dir))
    n_files = sum(1 for f in results_dir.rglob("*") if f.is_file())
    print(f"[sync_outputs] {n_files} files consolidated under {results_dir}, zipped to {archive_path}")


sync_outputs()  # harmless no-op now (nothing produced yet), confirms the function works before training starts

[sync_outputs] 0 files consolidated under /kaggle/working/outputs, zipped to /kaggle/working/new_way_results.zip


## Training -- 16 combos, cheapest architecture first

Each script's `model_name` carries a `_new_way` suffix so these results never collide with any
existing `model_name` in `classification/outputs/` (03_tech_stack_and_rules.md rule #3). Same
protocol as every other combo trained through this shared engine -- nothing about the search
itself changed, only the architecture roster.

In [8]:
# new_way 1/16 -- efficientnet_b3, palpebral
!python classification/new_way/train_efficientnet_b3_palpebral_new_way.py
sync_outputs()

Using device: cuda
Architecture: efficientnet_b3
Tissue type: palpebral
Model name: efficientnet_b3_palpebral_new_way
[I 2026-08-08 21:36:02,413] A new study created in memory with name: no-name-bd95fe29-ff7a-40aa-a3dd-d5753cb127ca
[efficientnet_b3_palpebral_new_way | Trial 0] New best overall val_f1=0.5143 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_efficientnet_b3_palpebral_new_way.pth
[efficientnet_b3_palpebral_new_way | Trial 0] Epoch  1/250 - train_loss=0.8023 val_loss=0.8074 val_acc=0.4848 val_f1=0.5143 India_acc=0.5714 Italy_acc=0.4211
[efficientnet_b3_palpebral_new_way | Trial 0] New best overall val_f1=0.5455 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_efficientnet_b3_palpebral_new_way.pth
[efficientnet_b3_palpebral_new_way | Trial 0] Epoch  2/250 - train_loss=0.7718 val_loss=0.8074 val_acc=0.5455 val_f1=0.5455 India_acc=0.5714 Italy_acc=0.5263
[efficientnet_b3_palpebral_new_way | Trial 0] Epoch  3/250

In [9]:
# new_way 2/16 -- efficientnet_b3, forniceal_palpebral
!python classification/new_way/train_efficientnet_b3_forniceal_palpebral_new_way.py
sync_outputs()

Using device: cuda
Architecture: efficientnet_b3
Tissue type: forniceal_palpebral
Model name: efficientnet_b3_forniceal_palpebral_new_way
[I 2026-08-08 21:42:06,200] A new study created in memory with name: no-name-dc830d30-99c2-453f-83da-22abd38e133b
[efficientnet_b3_forniceal_palpebral_new_way | Trial 0] New best overall val_f1=0.6250 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_efficientnet_b3_forniceal_palpebral_new_way.pth
[efficientnet_b3_forniceal_palpebral_new_way | Trial 0] Epoch  1/250 - train_loss=0.8055 val_loss=0.7833 val_acc=0.6129 val_f1=0.6250 India_acc=0.5714 Italy_acc=0.6471
[efficientnet_b3_forniceal_palpebral_new_way | Trial 0] Epoch  2/250 - train_loss=0.7700 val_loss=0.7799 val_acc=0.6452 val_f1=0.6207 India_acc=0.7143 Italy_acc=0.5882
[efficientnet_b3_forniceal_palpebral_new_way | Trial 0] Epoch  3/250 - train_loss=0.7414 val_loss=0.7779 val_acc=0.5806 val_f1=0.3810 India_acc=0.3571 Italy_acc=0.7647
[efficientnet_b3_forniceal_

In [10]:
# new_way 3/16 -- efficientnet_b4, palpebral
!python classification/new_way/train_efficientnet_b4_palpebral_new_way.py
sync_outputs()

Using device: cuda
Architecture: efficientnet_b4
Tissue type: palpebral
Model name: efficientnet_b4_palpebral_new_way
[I 2026-08-08 21:49:23,871] A new study created in memory with name: no-name-98bfcb34-964a-4b04-9eed-3149daa23e28
[efficientnet_b4_palpebral_new_way | Trial 0] New best overall val_f1=0.0000 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_efficientnet_b4_palpebral_new_way.pth
[efficientnet_b4_palpebral_new_way | Trial 0] Epoch  1/250 - train_loss=0.8053 val_loss=0.8084 val_acc=0.5758 val_f1=0.0000 India_acc=0.2857 Italy_acc=0.7895
[efficientnet_b4_palpebral_new_way | Trial 0] Epoch  2/250 - train_loss=0.7890 val_loss=0.8072 val_acc=0.5455 val_f1=0.0000 India_acc=0.2857 Italy_acc=0.7368
[efficientnet_b4_palpebral_new_way | Trial 0] New best overall val_f1=0.4211 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_efficientnet_b4_palpebral_new_way.pth
[efficientnet_b4_palpebral_new_way | Trial 0] Epoch  3/250

In [11]:
# new_way 4/16 -- efficientnet_b4, forniceal_palpebral
!python classification/new_way/train_efficientnet_b4_forniceal_palpebral_new_way.py
sync_outputs()

Using device: cuda
Architecture: efficientnet_b4
Tissue type: forniceal_palpebral
Model name: efficientnet_b4_forniceal_palpebral_new_way
[I 2026-08-08 22:06:44,261] A new study created in memory with name: no-name-6239453d-d469-47f0-b026-8a544529c5b5
[efficientnet_b4_forniceal_palpebral_new_way | Trial 0] New best overall val_f1=0.6667 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_efficientnet_b4_forniceal_palpebral_new_way.pth
[efficientnet_b4_forniceal_palpebral_new_way | Trial 0] Epoch  1/250 - train_loss=0.7979 val_loss=0.7945 val_acc=0.7419 val_f1=0.6667 India_acc=0.7143 Italy_acc=0.7647
[efficientnet_b4_forniceal_palpebral_new_way | Trial 0] Epoch  2/250 - train_loss=0.7842 val_loss=0.7917 val_acc=0.5161 val_f1=0.6154 India_acc=0.6429 Italy_acc=0.4118
[efficientnet_b4_forniceal_palpebral_new_way | Trial 0] Epoch  3/250 - train_loss=0.7730 val_loss=0.7841 val_acc=0.5806 val_f1=0.6667 India_acc=0.6429 Italy_acc=0.5294
[efficientnet_b4_forniceal_

In [12]:
# new_way 5/16 -- maxvit_t, palpebral
!python classification/new_way/train_maxvit_t_palpebral_new_way.py
sync_outputs()

Using device: cuda
Architecture: maxvit_t
Tissue type: palpebral
Model name: maxvit_t_palpebral_new_way
[I 2026-08-08 22:16:17,902] A new study created in memory with name: no-name-10fad7e2-d84c-4179-9d2f-e048ff1ffbf7
[maxvit_t_palpebral_new_way | Trial 0] New best overall val_f1=0.4138 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_maxvit_t_palpebral_new_way.pth
[maxvit_t_palpebral_new_way | Trial 0] Epoch  1/250 - train_loss=0.8194 val_loss=0.8118 val_acc=0.4848 val_f1=0.4138 India_acc=0.3571 Italy_acc=0.5789
[maxvit_t_palpebral_new_way | Trial 0] Epoch  2/250 - train_loss=0.7632 val_loss=0.7998 val_acc=0.5152 val_f1=0.1111 India_acc=0.2857 Italy_acc=0.6842
[maxvit_t_palpebral_new_way | Trial 0] Epoch  3/250 - train_loss=0.7339 val_loss=0.7886 val_acc=0.6364 val_f1=0.3333 India_acc=0.4286 Italy_acc=0.7895
[maxvit_t_palpebral_new_way | Trial 0] New best overall val_f1=0.5217 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints

In [13]:
# new_way 6/16 -- maxvit_t, forniceal_palpebral
!python classification/new_way/train_maxvit_t_forniceal_palpebral_new_way.py
sync_outputs()

Using device: cuda
Architecture: maxvit_t
Tissue type: forniceal_palpebral
Model name: maxvit_t_forniceal_palpebral_new_way
[I 2026-08-08 22:36:46,133] A new study created in memory with name: no-name-61bb6b20-37ae-4b04-b0e9-8cd8d3c6425d
[maxvit_t_forniceal_palpebral_new_way | Trial 0] New best overall val_f1=0.0000 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_maxvit_t_forniceal_palpebral_new_way.pth
[maxvit_t_forniceal_palpebral_new_way | Trial 0] Epoch  1/250 - train_loss=0.8142 val_loss=0.8094 val_acc=0.5161 val_f1=0.0000 India_acc=0.2857 Italy_acc=0.7059
[maxvit_t_forniceal_palpebral_new_way | Trial 0] Epoch  2/250 - train_loss=0.7634 val_loss=0.8072 val_acc=0.5484 val_f1=0.0000 India_acc=0.2857 Italy_acc=0.7647
[maxvit_t_forniceal_palpebral_new_way | Trial 0] New best overall val_f1=0.2222 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_maxvit_t_forniceal_palpebral_new_way.pth
[maxvit_t_forniceal_palpebral_new_

In [14]:
# new_way 7/16 -- maxvit_small, palpebral
!python classification/new_way/train_maxvit_small_palpebral_new_way.py
sync_outputs()

Using device: cuda
Architecture: maxvit_small
Tissue type: palpebral
Model name: maxvit_small_palpebral_new_way
[I 2026-08-08 22:46:08,324] A new study created in memory with name: no-name-fd2fb805-1fc8-4217-be89-45d1c7566f99
[maxvit_small_palpebral_new_way | Trial 0] New best overall val_f1=0.0000 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_maxvit_small_palpebral_new_way.pth
[maxvit_small_palpebral_new_way | Trial 0] Epoch  1/250 - train_loss=0.8003 val_loss=0.7942 val_acc=0.5758 val_f1=0.0000 India_acc=0.2857 Italy_acc=0.7895
[maxvit_small_palpebral_new_way | Trial 0] Epoch  2/250 - train_loss=0.7310 val_loss=0.7850 val_acc=0.5758 val_f1=0.0000 India_acc=0.2857 Italy_acc=0.7895
[maxvit_small_palpebral_new_way | Trial 0] New best overall val_f1=0.4000 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_maxvit_small_palpebral_new_way.pth
[maxvit_small_palpebral_new_way | Trial 0] Epoch  3/250 - train_loss=0.6859 val_lo

In [15]:
# new_way 8/16 -- maxvit_small, forniceal_palpebral
!python classification/new_way/train_maxvit_small_forniceal_palpebral_new_way.py
sync_outputs()

Using device: cuda
Architecture: maxvit_small
Tissue type: forniceal_palpebral
Model name: maxvit_small_forniceal_palpebral_new_way
[I 2026-08-08 23:18:12,361] A new study created in memory with name: no-name-47722ca3-48ec-42e4-a1e1-26cab6ebd320
[maxvit_small_forniceal_palpebral_new_way | Trial 0] New best overall val_f1=0.5946 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_maxvit_small_forniceal_palpebral_new_way.pth
[maxvit_small_forniceal_palpebral_new_way | Trial 0] Epoch  1/250 - train_loss=0.8195 val_loss=0.8014 val_acc=0.5161 val_f1=0.5946 India_acc=0.5000 Italy_acc=0.5294
[maxvit_small_forniceal_palpebral_new_way | Trial 0] New best overall val_f1=0.6341 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_maxvit_small_forniceal_palpebral_new_way.pth
[maxvit_small_forniceal_palpebral_new_way | Trial 0] Epoch  2/250 - train_loss=0.7692 val_loss=0.7951 val_acc=0.5161 val_f1=0.6341 India_acc=0.6429 Italy_acc=0.4118
[m

In [16]:
# new_way 9/16 -- regnet_y_16gf, palpebral
!python classification/new_way/train_regnet_y_16gf_palpebral_new_way.py
sync_outputs()

Using device: cuda
Architecture: regnet_y_16gf
Tissue type: palpebral
Model name: regnet_y_16gf_palpebral_new_way
[I 2026-08-08 23:31:11,608] A new study created in memory with name: no-name-13aeafb7-cbaa-4e87-a912-fa43f70078db
[regnet_y_16gf_palpebral_new_way | Trial 0] New best overall val_f1=0.5957 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_regnet_y_16gf_palpebral_new_way.pth
[regnet_y_16gf_palpebral_new_way | Trial 0] Epoch  1/250 - train_loss=0.8552 val_loss=0.8427 val_acc=0.4242 val_f1=0.5957 India_acc=0.7143 Italy_acc=0.2105
[regnet_y_16gf_palpebral_new_way | Trial 0] Epoch  2/250 - train_loss=0.7640 val_loss=0.7592 val_acc=0.5758 val_f1=0.1250 India_acc=0.2857 Italy_acc=0.7895
[regnet_y_16gf_palpebral_new_way | Trial 0] New best overall val_f1=0.7778 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_regnet_y_16gf_palpebral_new_way.pth
[regnet_y_16gf_palpebral_new_way | Trial 0] Epoch  3/250 - train_loss=0.71

In [17]:
# new_way 10/16 -- regnet_y_16gf, forniceal_palpebral
!python classification/new_way/train_regnet_y_16gf_forniceal_palpebral_new_way.py
sync_outputs()

Using device: cuda
Architecture: regnet_y_16gf
Tissue type: forniceal_palpebral
Model name: regnet_y_16gf_forniceal_palpebral_new_way
[I 2026-08-09 00:16:37,014] A new study created in memory with name: no-name-82752ada-623e-4240-b7b2-69f1f0315e13
[regnet_y_16gf_forniceal_palpebral_new_way | Trial 0] New best overall val_f1=0.6222 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_regnet_y_16gf_forniceal_palpebral_new_way.pth
[regnet_y_16gf_forniceal_palpebral_new_way | Trial 0] Epoch  1/250 - train_loss=0.8244 val_loss=0.8754 val_acc=0.4516 val_f1=0.6222 India_acc=0.7143 Italy_acc=0.2353
[regnet_y_16gf_forniceal_palpebral_new_way | Trial 0] Epoch  2/250 - train_loss=0.7657 val_loss=0.8540 val_acc=0.5484 val_f1=0.0000 India_acc=0.2857 Italy_acc=0.7647
[regnet_y_16gf_forniceal_palpebral_new_way | Trial 0] New best overall val_f1=0.7000 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_regnet_y_16gf_forniceal_palpebral_new_wa

In [18]:
# new_way 11/16 -- convnext_base, palpebral
!python classification/new_way/train_convnext_base_palpebral_new_way.py
sync_outputs()

Using device: cuda
Architecture: convnext_base
Tissue type: palpebral
Model name: convnext_base_palpebral_new_way
[I 2026-08-09 01:04:52,937] A new study created in memory with name: no-name-2207fdab-ab15-4482-8d18-0b3533d74a01
[convnext_base_palpebral_new_way | Trial 0] New best overall val_f1=0.7500 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_convnext_base_palpebral_new_way.pth
[convnext_base_palpebral_new_way | Trial 0] Epoch  1/250 - train_loss=0.8019 val_loss=0.7234 val_acc=0.8182 val_f1=0.7500 India_acc=0.8571 Italy_acc=0.7895
[convnext_base_palpebral_new_way | Trial 0] New best overall val_f1=0.8000 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_convnext_base_palpebral_new_way.pth
[convnext_base_palpebral_new_way | Trial 0] Epoch  2/250 - train_loss=0.7486 val_loss=0.6783 val_acc=0.7879 val_f1=0.8000 India_acc=0.8571 Italy_acc=0.7368
[convnext_base_palpebral_new_way | Trial 0] New best overall val_f1=0.8276

In [19]:
# new_way 12/16 -- convnext_base, forniceal_palpebral
!python classification/new_way/train_convnext_base_forniceal_palpebral_new_way.py
sync_outputs()

Using device: cuda
Architecture: convnext_base
Tissue type: forniceal_palpebral
Model name: convnext_base_forniceal_palpebral_new_way
[I 2026-08-09 02:10:03,132] A new study created in memory with name: no-name-ea0cd1e6-6887-453c-bd93-e7c5e838ff1e
[convnext_base_forniceal_palpebral_new_way | Trial 0] New best overall val_f1=0.6667 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_convnext_base_forniceal_palpebral_new_way.pth
[convnext_base_forniceal_palpebral_new_way | Trial 0] Epoch  1/250 - train_loss=0.8267 val_loss=0.7818 val_acc=0.6774 val_f1=0.6667 India_acc=0.6429 Italy_acc=0.7059
[convnext_base_forniceal_palpebral_new_way | Trial 0] Epoch  2/250 - train_loss=0.7818 val_loss=0.7760 val_acc=0.5484 val_f1=0.6667 India_acc=0.7143 Italy_acc=0.4118
[convnext_base_forniceal_palpebral_new_way | Trial 0] Epoch  3/250 - train_loss=0.7518 val_loss=0.7705 val_acc=0.5161 val_f1=0.4828 India_acc=0.5000 Italy_acc=0.5294
[convnext_base_forniceal_palpebral_new_wa

In [20]:
# new_way 13/16 -- coatnet_3, palpebral
!python classification/new_way/train_coatnet_3_palpebral_new_way.py
sync_outputs()

Using device: cuda
Architecture: coatnet_3
Tissue type: palpebral
Model name: coatnet_3_palpebral_new_way
[I 2026-08-09 02:29:53,028] A new study created in memory with name: no-name-2a0d0e21-6f9f-4b62-9855-d8f5a542e2ef
[coatnet_3_palpebral_new_way | Trial 0] New best overall val_f1=0.6897 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_coatnet_3_palpebral_new_way.pth
[coatnet_3_palpebral_new_way | Trial 0] Epoch  1/250 - train_loss=1.4932 val_loss=0.6878 val_acc=0.7273 val_f1=0.6897 India_acc=0.7857 Italy_acc=0.6842
[coatnet_3_palpebral_new_way | Trial 0] Epoch  2/250 - train_loss=0.8427 val_loss=1.0138 val_acc=0.5455 val_f1=0.6512 India_acc=0.7143 Italy_acc=0.4211
[coatnet_3_palpebral_new_way | Trial 0] Epoch  3/250 - train_loss=0.8430 val_loss=0.9648 val_acc=0.5152 val_f1=0.6364 India_acc=0.7143 Italy_acc=0.3684
[coatnet_3_palpebral_new_way | Trial 0] Epoch  4/250 - train_loss=0.8330 val_loss=1.6189 val_acc=0.4242 val_f1=0.5957 India_acc=0.7143 Ital

In [21]:
# new_way 14/16 -- coatnet_3, forniceal_palpebral
!python classification/new_way/train_coatnet_3_forniceal_palpebral_new_way.py
sync_outputs()

Using device: cuda
Architecture: coatnet_3
Tissue type: forniceal_palpebral
Model name: coatnet_3_forniceal_palpebral_new_way
[I 2026-08-09 02:52:40,670] A new study created in memory with name: no-name-abf62f93-5f2b-4865-80f5-115ef0628bc6
[coatnet_3_forniceal_palpebral_new_way | Trial 0] New best overall val_f1=0.6154 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_coatnet_3_forniceal_palpebral_new_way.pth
[coatnet_3_forniceal_palpebral_new_way | Trial 0] Epoch  1/250 - train_loss=1.2345 val_loss=0.9487 val_acc=0.5161 val_f1=0.6154 India_acc=0.6429 Italy_acc=0.4118
[coatnet_3_forniceal_palpebral_new_way | Trial 0] New best overall val_f1=0.6190 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_coatnet_3_forniceal_palpebral_new_way.pth
[coatnet_3_forniceal_palpebral_new_way | Trial 0] Epoch  2/250 - train_loss=0.9181 val_loss=1.0598 val_acc=0.4839 val_f1=0.6190 India_acc=0.7143 Italy_acc=0.2941
[coatnet_3_forniceal_palpe

In [22]:
# new_way 15/16 -- convnext_large, palpebral
!python classification/new_way/train_convnext_large_palpebral_new_way.py
sync_outputs()

Using device: cuda
Architecture: convnext_large
Tissue type: palpebral
Model name: convnext_large_palpebral_new_way
[I 2026-08-09 03:15:11,968] A new study created in memory with name: no-name-9f618cf1-4079-41dd-8f2a-360dde15c56c
[convnext_large_palpebral_new_way | Trial 0] New best overall val_f1=0.6222 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_convnext_large_palpebral_new_way.pth
[convnext_large_palpebral_new_way | Trial 0] Epoch  1/250 - train_loss=0.8171 val_loss=0.7619 val_acc=0.4848 val_f1=0.6222 India_acc=0.7143 Italy_acc=0.3158
[convnext_large_palpebral_new_way | Trial 0] New best overall val_f1=0.6364 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_convnext_large_palpebral_new_way.pth
[convnext_large_palpebral_new_way | Trial 0] Epoch  2/250 - train_loss=0.7671 val_loss=0.7066 val_acc=0.7576 val_f1=0.6364 India_acc=0.7143 Italy_acc=0.7895
[convnext_large_palpebral_new_way | Trial 0] New best overall val_

In [23]:
# new_way 16/16 -- convnext_large, forniceal_palpebral
!python classification/new_way/train_convnext_large_forniceal_palpebral_new_way.py
sync_outputs()

Using device: cuda
Architecture: convnext_large
Tissue type: forniceal_palpebral
Model name: convnext_large_forniceal_palpebral_new_way
[I 2026-08-09 05:19:58,095] A new study created in memory with name: no-name-6c694123-8697-486b-a500-3db1bcd2c4f2
[convnext_large_forniceal_palpebral_new_way | Trial 0] New best overall val_f1=0.5185 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_convnext_large_forniceal_palpebral_new_way.pth
[convnext_large_forniceal_palpebral_new_way | Trial 0] Epoch  1/250 - train_loss=0.8087 val_loss=0.7957 val_acc=0.5806 val_f1=0.5185 India_acc=0.5714 Italy_acc=0.5882
[convnext_large_forniceal_palpebral_new_way | Trial 0] New best overall val_f1=0.6667 -> saved /kaggle/working/eyes-defy-anemia/classification/outputs/checkpoints/best_convnext_large_forniceal_palpebral_new_way.pth
[convnext_large_forniceal_palpebral_new_way | Trial 0] Epoch  2/250 - train_loss=0.7992 val_loss=0.7966 val_acc=0.5484 val_f1=0.6667 India_acc=0.7857 Ita

## Done -- what to download

Everything is consolidated at `/kaggle/working/outputs/` (checkpoints, logs, plots for whichever
combos completed) and zipped to `/kaggle/working/new_way_results.zip`. Both are visible in this
notebook version's **Output** tab once you Save Version -> Save & Run All -- download the zip
directly from there, or browse the folder for individual files.

Same downstream step as every prior batch in this project: extract the zip, run
`classification/v2_clean_scripts/organize_and_compare.py <path-to-extracted-outputs>` (or a
dedicated copy for this roster) to reorganize into per-combo folders and build a comparison table.

In [24]:
print('Final contents of /kaggle/working/outputs:')
for f in sorted(Path('/kaggle/working/outputs').rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to("/kaggle/working/outputs")}  ({f.stat().st_size / 1e6:.2f} MB)')

zip_path = Path('/kaggle/working/new_way_results.zip')
print(f"\nZip archive: {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)")

Final contents of /kaggle/working/outputs:
  checkpoints/best_coatnet_3_forniceal_palpebral_new_way.pth  (654.92 MB)
  checkpoints/best_coatnet_3_palpebral_new_way.pth  (654.91 MB)
  checkpoints/best_convnext_base_forniceal_palpebral_new_way.pth  (350.41 MB)
  checkpoints/best_convnext_base_palpebral_new_way.pth  (350.41 MB)
  checkpoints/best_convnext_large_forniceal_palpebral_new_way.pth  (785.07 MB)
  checkpoints/best_convnext_large_palpebral_new_way.pth  (785.06 MB)
  checkpoints/best_efficientnet_b3_forniceal_palpebral_new_way.pth  (43.38 MB)
  checkpoints/best_efficientnet_b3_palpebral_new_way.pth  (43.37 MB)
  checkpoints/best_efficientnet_b4_forniceal_palpebral_new_way.pth  (70.99 MB)
  checkpoints/best_efficientnet_b4_palpebral_new_way.pth  (70.98 MB)
  checkpoints/best_maxvit_small_forniceal_palpebral_new_way.pth  (273.17 MB)
  checkpoints/best_maxvit_small_palpebral_new_way.pth  (273.17 MB)
  checkpoints/best_maxvit_t_forniceal_palpebral_new_way.pth  (122.52 MB)
  checkpoint